# Pengukuran Jarak Data Campuran

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import pdist, squareform

# Load data
df = pd.read_csv('google_play_store_apps_famous.csv')

# Memilih 5 aplikasi pertama
cols = ['App', 'Category', 'Rating', 'Price_$', 'Content_Rating']
df_sample = df[cols].head(5).copy()

print("Dataset Sampel:")
df_sample

## Normalisasi Data
kita tidak bisa langsung menghitung jarak jika tipe datanya berbeda. Kita harus melakukan Normalisasi agar semua nilai berada di rentang $[0, 1]$.

**A. Atribut Ordinal (Content_Rating)**
Data ordinal harus diubah menjadi peringkat (rank), lalu dipetakan ke skala 
$[0, 1]$ dengan rumus:

$$z_{if} = \frac{r_{if} - 1}{M_f - 1}$$

- App_1 (Teen=2): $(2 - 1) / (3 - 1) = 1 / 2 = \mathbf{0.5}$
- App_2 (Mature=3): $(3 - 1) / (3 - 1) = 2 / 2 = \mathbf{1.0}$

Di mana $M_f$ adalah peringkat tertinggi.

**B. Atribut Numerik (Rating, Price_$)**
Data numerik harus dinormalisasi agar variabel dengan angka besar (seperti harga) tidak mendominasi variabel kecil (seperti rating).

- Rating App_1: $(3.7 - 2.5) / (5.0 - 2.5) = 1.2 / 2.5 = \mathbf{0.48}$
- Rating App_2: $(5.0 - 2.5) / (5.0 - 2.5) = 2.5 / 2.5 = \mathbf{1.0}$
- Price App_1: $(1.99 - 0.0) / (2.99 - 0.0) \approx \mathbf{0.6655}$
- Price App_2: $(0.99 - 0.0) / (2.99 - 0.0) \approx \mathbf{0.3311}$

Hasil Normalisasi

- App_1: $[0.48, 0.6655, 0.5]$
- App_2: $[1.0, 0.3311, 1.0]$

In [ ]:
# Menentukan(Pengubahan ordinal) urutan: Everyone < Teen < Mature 17+
rank_map = {'Everyone': 1, 'Teen': 2, 'Mature 17+': 3}
df_sample['Rank_Ordinal'] = df_sample['Content_Rating'].map(rank_map)

# Normalisasi Ordinal ke skala [0,1]
max_rank = 3
df_sample['Ordinal_Norm'] = (df_sample['Rank_Ordinal'] - 1) / (max_rank - 1)

# Menggunakan MinMaxScaler untuk mengubah data ke rentang [0, 1]
scaler = MinMaxScaler()
df_sample[['Rating_Norm', 'Price_Norm']] = scaler.fit_transform(df_sample[['Rating', 'Price_$']])

print("Hasil Transformasi ke Skala [0, 1]:")
df_sample[['App', 'Rating_Norm', 'Price_Norm', 'Ordinal_Norm']]

## Implementasi Manhattan Distance ($L_1$)
Manhattan Distance (City Block) dihitung dengan menjumlahkan selisih absolut dari setiap atribut.

$$d(i, j) = |x_{i1} - x_{j1}| + |x_{i2} - x_{j2}| + \dots + |x_{ip} - x_{jp}|$$

Perhitungan Jarak App_1 ke App_2:
- Selisih Rating: $|0.48 - 1.0| = 0.52$
- Selisih Price: $|0.6655 - 0.3311| = 0.3344$
- Selisih Ordinal: $|0.5 - 1.0| = 0.5$
- Total Manhattan: $0.52 + 0.3344 + 0.5 = \mathbf{1.3544}$

In [ ]:
# Mengambil fitur yang sudah dinormalisasi
features = ['Rating_Norm', 'Price_Norm', 'Ordinal_Norm']
data_matrix = df_sample[features].values

# Menghitung Manhattan Distance
dist_l1 = pdist(data_matrix, metric='cityblock')

# Mengubah hasil menjadi Matriks Jarak 
matrix_manhattan = pd.DataFrame(squareform(dist_l1), 
                                columns=df_sample['App'], 
                                index=df_sample['App'])

print("Matriks Jarak Manhattan:")
matrix_manhattan

## Implementasi Euclidean Distance ($L_2$)
Euclidean Distance adalah jarak "garis lurus" antar dua titik di ruang multidimensi.

$$d(i, j) = \sqrt{(x_{i1} - x_{j1})^2 + (x_{i2} - x_{j2})^2 + \dots + (x_{ip} - x_{jp})^2}$$

Perhitungan Jarak App_1 ke App_2:
- Kuadrat Selisih Rating: $(0.52)^2 = 0.2704$
- Kuadrat Selisih Price: $(0.3344)^2 = 0.1118$
- Kuadrat Selisih Ordinal: $(0.5)^2 = 0.25$
- Total Euclidean: $\sqrt{0.2704 + 0.1118 + 0.25} = \sqrt{0.6322} \approx \mathbf{0.7951}$

In [ ]:
# Menghitung Euclidean Distance
dist_l2 = pdist(data_matrix, metric='euclidean')

# Mengubah hasil menjadi Matriks Jarak
matrix_euclidean = pd.DataFrame(squareform(dist_l2), 
                                columns=df_sample['App'], 
                                index=df_sample['App'])

print("Matriks Jarak Euclidean:")
matrix_euclidean

## Menggabungkan Atribut Nominal 
Untuk menghitung jarak total pada Atribut Campuran, PPT menyarankan penggunaan pembobotan. Untuk atribut Nominal (Category), jika nilainya sama maka jaraknya $0$, jika berbeda jaraknya $1$.

Gabungkan atribut Nominal (Category) dengan hasil Manhattan tadi.

- Atribut Nominal (Category): App_1 (HEALTH) vs App_2 (LIFESTYLE). Karena berbeda, maka jaraknya = 1.
- Atribut Lain (Rating, Price, Content): Kita gunakan hasil jumlah selisih Manhattan = 1.3544.

Pergitungan Jarak Total :

Jumlah atribut ($p$) = 4 (Category, Rating, Price, Content).

$$Jarak = \frac{1 (\text{Nominal}) + 1.3544 (\text{Manhattan})}{4}$$$$Jarak = \frac{2.3544}{4} = \mathbf{0.5886}$$

In [ ]:
def hitung_jarak_campuran(idx1, idx2, df):
    # Jarak Nominal (Kategori)
    d_nominal = 0 if df.loc[idx1, 'Category'] == df.loc[idx2, 'Category'] else 1
    
    # Jarak Numerik & Ordinal (Manhattan)
    # Kita ambil selisih dari kolom yang sudah dinormalisasi
    dist_vals = np.abs(df.loc[idx1, features] - df.loc[idx2, features])
    d_others = dist_vals.sum()
    
    # Rata-rata Jarak
    # Total atribut = 1 (nominal) + 3 (numerik/ordinal) = 4
    jarak_total = (d_nominal + d_others) / 4
    return jarak_total

# Contoh: Hitung jarak antara App_1 dan App_2
hasil_campuran = hitung_jarak_campuran(0, 1, df_sample)
print(f"Jarak Campuran Total (Gower) antara {df_sample.loc[0,'App']} dan {df_sample.loc[1,'App']}: {hasil_campuran:.4f}")

## Kesimpulan
1. Matriks Jarak: Hasil di atas menunjukkan tingkat ketidaksamaan (dissimilarity). Nilai 0 pada diagonal menunjukkan bahwa objek tersebut identik.

2. Manhattan vs Euclidean: Manhattan memberikan bobot yang lebih linear pada perbedaan setiap atribut, sedangkan Euclidean memberikan penalti lebih besar pada perbedaan yang ekstrim karena adanya pangkat dua.

3. Normalisasi: Tanpa normalisasi di langkah ke-2, hasil perhitungan akan salah karena skala angka yang tidak seimbang.